<a href="https://colab.research.google.com/github/devcodes28/AI-ML-Lab-24CSP503/blob/main/AI_ML_Lab_Universal_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI/ML Lab Exam — Universal Tabular ML Notebook

A reusable starting point for common **supervised tabular tasks**: classification, regression, and an artificial neural network (MLP).

**Important:** no single notebook can automatically solve literally every possible task. NLP, images, clustering, and time-series forecasting may need different workflows. Read the exam question, set the target and task, and tailor the metric/model accordingly.

## Exam workflow
1. Load the provided CSV.
2. Identify the column to predict (target).
3. Set `TARGET_COLUMN` and `TASK_OVERRIDE` in the configuration cell.
4. Run cells in order and inspect outputs.
5. Explain preprocessing, train/test split, model comparison, metrics, and limitations.

## 1. Imports and reproducibility

In [ ]:
import warnings, os, re, sys
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, mean_absolute_error,
    mean_squared_error, r2_score)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, RandomForestRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.base import clone

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

## 2. Configuration — edit these for the exam

- `DATA_PATH`: CSV path; leave `None` to upload in Google Colab.
- `TARGET_COLUMN`: exact name of the column to predict.
- `TASK_OVERRIDE`: `"auto"`, `"classification"`, or `"regression"`. Follow the exam wording rather than trusting automatic inference.
- `DROP_COLUMNS`: known identifiers or columns the question says to exclude.
- Outlier clipping is optional and off by default: outliers may be valid observations.

In [ ]:
DATA_PATH = None
TARGET_COLUMN = None
TASK_OVERRIDE = "auto"
DROP_COLUMNS = []
TEST_SIZE = 0.20
CLIP_OUTLIERS = False
LOWER_QUANTILE = 0.01
UPPER_QUANTILE = 0.99

## 3. Load the dataset

In [ ]:
if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded; set DATA_PATH.")
        DATA_PATH = next(iter(uploaded.keys()))
    except ImportError:
        DATA_PATH = input("Enter path to CSV: ").strip()

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]
print("Shape:", df.shape)
display(df.head())
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (100*df.isna().mean()).round(2),
    "unique": df.nunique(dropna=False)
}).sort_values("missing_%", ascending=False))

## 4. Initial inspection and data quality

In [ ]:
print("Columns:", df.columns.tolist())
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(df.describe(include="all").T)
missing_pct = (df.isna().mean()*100).sort_values(ascending=False)
if (missing_pct > 0).any():
    missing_pct[missing_pct > 0].plot(kind="bar", figsize=(10,4), title="Missing values (%)")
    plt.tight_layout(); plt.show()
num = df.select_dtypes(include=np.number)
if not num.empty:
    num.hist(figsize=(12,8), bins=25)
    plt.tight_layout(); plt.show()

## 5. Basic cleaning and data-type mismatch handling

This step standardises common missing tokens, removes exact duplicate rows, and converts object columns that are overwhelmingly numeric. Inspect the conversion log: automatic conversion is a heuristic, not a substitute for domain knowledge.

In [ ]:
missing_tokens = {"", " ", "NA", "N/A", "na", "n/a", "NULL", "null", "None", "none", "?", "-", "unknown", "Unknown"}
for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].map(lambda x: np.nan if isinstance(x, str) and x.strip() in missing_tokens else x)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Exact duplicates removed:", before-len(df))

conversion_log = []
for c in df.select_dtypes(include=["object", "string"]).columns:
    s = df[c].astype("string").str.strip()
    candidate = s.str.replace(",", "", regex=False).str.replace("$", "", regex=False).str.replace("%", "", regex=False)
    parsed = pd.to_numeric(candidate, errors="coerce")
    n = int(s.notna().sum())
    ratio = parsed.notna().sum()/n if n else 0
    if n and ratio >= 0.90:
        df[c] = parsed
        conversion_log.append((c, round(float(ratio), 3)))
print("Object-to-numeric conversions (column, parse ratio):", conversion_log)
print("Potential date/time columns to inspect:", [c for c in df.columns if re.search(r"date|time|timestamp", str(c), re.I)])

present = [c for c in DROP_COLUMNS if c in df.columns]
if present:
    df = df.drop(columns=present)
print("Current shape:", df.shape)
display(df.head())

## 6. Choose target and task

Set `TARGET_COLUMN` above, then run this cell. Rows with missing target values are excluded because supervised learning needs known labels. Numeric targets with few unique values can be either classes or numbers, so automatic task detection is only a guess.

In [ ]:
if TARGET_COLUMN is None:
    print("Target candidates:")
    for c in df.columns:
        print(f"{c!r}: dtype={df[c].dtype}, unique={df[c].nunique(dropna=True)}, examples={df[c].dropna().astype(str).head(4).tolist()}")
    raise ValueError("Set TARGET_COLUMN in the configuration cell and rerun.")
if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Target not found. Available columns: {df.columns.tolist()}")

data = df.loc[df[TARGET_COLUMN].notna()].copy()
y = data[TARGET_COLUMN].copy()
X = data.drop(columns=[TARGET_COLUMN]).copy()
if TASK_OVERRIDE not in {"auto", "classification", "regression"}:
    raise ValueError("TASK_OVERRIDE must be auto, classification, or regression.")
if TASK_OVERRIDE == "auto":
    TASK = "classification" if (y.dtype == "object" or str(y.dtype) in ["category","bool","string"] or y.nunique() <= 15) else "regression"
else:
    TASK = TASK_OVERRIDE

print("Target:", TARGET_COLUMN, "| Task:", TASK, "| X shape:", X.shape)
display(y.value_counts().head(20).to_frame("count") if TASK == "classification" else y.describe().to_frame("summary"))
if X.shape[1] == 0:
    raise ValueError("No predictor columns remain.")

## 7. Train/test split

Split before fitting imputation, scaling, encoding, or outlier thresholds to avoid data leakage. If the question involves future/time-series prediction, replace this random split with a chronological split.

In [ ]:
stratify_y = None
if TASK == "classification":
    counts = y.value_counts()
    if len(counts) > 1 and counts.min() >= 2:
        stratify_y = y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify_y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 8. Preprocessing pipeline

Numeric predictors use median imputation and scaling. Categorical predictors use most-frequent imputation and one-hot encoding. The preprocessing is inside each model pipeline, so it is fitted only on training data. Optional quantile clipping is learned from training data only.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

class QuantileClipper:
    # Clip values to training-set quantiles; intended only for numeric predictors.
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper
    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(arr, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(arr, self.upper, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_bounds_, self.upper_bounds_)

numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
if CLIP_OUTLIERS:
    numeric_steps.append(("clipper", QuantileClipper(LOWER_QUANTILE, UPPER_QUANTILE)))
numeric_steps.append(("scaler", StandardScaler()))
numeric_pipeline = Pipeline(numeric_steps)
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
], remainder="drop")

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)
print("Outlier clipping:", CLIP_OUTLIERS)

## 9. Candidate models, including an Artificial Neural Network

`MLPClassifier` and `MLPRegressor` are feed-forward artificial neural networks. We compare them with simple baselines and traditional ML models. For very small datasets or extremely wide data, the ANN may not be the best model.

In [ ]:
if TASK == "classification":
    models = {
        "Baseline (most frequent)": DummyClassifier(strategy="most_frequent"),
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(n_estimators=250, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
        "Artificial Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(128,64), activation="relu", early_stopping=True, max_iter=400, random_state=RANDOM_STATE)
    }
else:
    models = {
        "Baseline (mean)": DummyRegressor(strategy="mean"),
        "Ridge Regression": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1),
        "Extra Trees": ExtraTreesRegressor(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1),
        "Artificial Neural Network (MLP)": MLPRegressor(hidden_layer_sizes=(128,64), activation="relu", early_stopping=True, max_iter=400, random_state=RANDOM_STATE)
    }
pipelines = {name: Pipeline([("preprocess", clone(preprocessor)), ("model", est)]) for name, est in models.items()}
print(list(pipelines))

## 10. Train and compare models

In [ ]:
fitted_models, rows = {}, []
for name, pipe in pipelines.items():
    try:
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        fitted_models[name] = pipe
        if TASK == "classification":
            rows.append({
                "Model": name,
                "Accuracy": accuracy_score(y_test, pred),
                "Balanced accuracy": balanced_accuracy_score(y_test, pred),
                "Precision weighted": precision_score(y_test, pred, average="weighted", zero_division=0),
                "Recall weighted": recall_score(y_test, pred, average="weighted", zero_division=0),
                "F1 weighted": f1_score(y_test, pred, average="weighted", zero_division=0)
            })
        else:
            rows.append({
                "Model": name,
                "MAE (lower better)": mean_absolute_error(y_test, pred),
                "RMSE (lower better)": mean_squared_error(y_test, pred)**0.5,
                "R2 (higher better)": r2_score(y_test, pred)
            })
    except Exception as e:
        print(f"Skipped {name}: {type(e).__name__}: {e}")

results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError("No model trained successfully. Check dataset, target, and data types.")
if TASK == "classification":
    results = results.sort_values(["F1 weighted", "Balanced accuracy"], ascending=False)
    best_name = results.iloc[0]["Model"]
else:
    results = results.sort_values(["RMSE (lower better)", "MAE (lower better)"])
    best_name = results.iloc[0]["Model"]
display(results.round(4))
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)
print("Selected model:", best_name)

## 11. Detailed evaluation of selected model

In [ ]:
if TASK == "classification":
    print(classification_report(y_test, best_pred, zero_division=0))
    model_step = best_model.named_steps["model"]
    labels = model_step.classes_ if hasattr(model_step, "classes_") else np.unique(y_test)
    ConfusionMatrixDisplay(confusion_matrix(y_test, best_pred, labels=labels), display_labels=labels).plot(xticks_rotation=45)
    plt.title("Confusion matrix — " + best_name)
    plt.tight_layout(); plt.show()
    if hasattr(best_model, "predict_proba"):
        try:
            proba = best_model.predict_proba(X_test)
            if len(model_step.classes_) == 2:
                positive_class = model_step.classes_[1]
                y_binary = (np.asarray(y_test) == positive_class).astype(int)
                print("ROC-AUC:", round(roc_auc_score(y_binary, proba[:,1]), 4))
            else:
                print("ROC-AUC OVR weighted:", round(roc_auc_score(y_test, proba, multi_class="ovr", average="weighted"), 4))
        except Exception as e:
            print("ROC-AUC unavailable:", e)
else:
    print("MAE:", round(mean_absolute_error(y_test, best_pred), 4))
    print("RMSE:", round(mean_squared_error(y_test, best_pred)**0.5, 4))
    print("R²:", round(r2_score(y_test, best_pred), 4))
    plt.figure(figsize=(6,6))
    plt.scatter(y_test, best_pred, alpha=0.7)
    low, high = min(np.min(y_test), np.min(best_pred)), max(np.max(y_test), np.max(best_pred))
    plt.plot([low,high], [low,high], "--")
    plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Actual vs predicted")
    plt.tight_layout(); plt.show()

## 12. Outlier inspection (IQR method)

IQR flags are a diagnostic heuristic, not proof of incorrect data. Verify whether extremes are errors or valid observations before removing or capping them. The main workflow does not delete rows because of outliers.

In [ ]:
numeric_all = df.select_dtypes(include=np.number)
if numeric_all.empty:
    print("No numeric columns for IQR analysis.")
else:
    report = []
    for c in numeric_all.columns:
        q1, q3 = numeric_all[c].quantile([0.25, 0.75])
        iqr = q3-q1
        count = 0 if pd.isna(iqr) or iqr == 0 else int(((numeric_all[c] < q1-1.5*iqr) | (numeric_all[c] > q3+1.5*iqr)).sum())
        report.append({"column":c, "IQR_flagged_count":count})
    display(pd.DataFrame(report).sort_values("IQR_flagged_count", ascending=False))

## 13. Predict new rows

In [ ]:
# Replace example values with a real row. Use exactly the same predictor column names as X.
# new_rows = pd.DataFrame([{c: np.nan for c in X.columns}])
# predictions = best_model.predict(new_rows)
# display(pd.DataFrame({"prediction": predictions}))
print("Create new_rows with the same columns as X, then run best_model.predict(new_rows).")

## 14. Save the trained pipeline (optional)

In [ ]:
# Uncomment if needed. The pipeline includes preprocessing + fitted estimator.
# import joblib
# joblib.dump(best_model, "best_ml_pipeline.joblib")
# print("Saved best_ml_pipeline.joblib")

## 15. Exam/viva checklist

- **Task:** explain why this is classification or regression.
- **Target/features:** name the label and predictors; remove IDs or leakage columns only when justified.
- **Cleaning:** report missing values, duplicate rows, type mismatches, and possible outliers.
- **Preprocessing:** median/mode imputation, scaling, one-hot encoding; explain why fitted preprocessing belongs inside the pipeline.
- **Split:** explain held-out test data and data leakage.
- **ANN:** MLP has hidden layers `(128, 64)`, ReLU activation, and early stopping; it is one candidate, not automatically the best.
- **Metrics:** classification uses confusion matrix, precision, recall, F1, balanced accuracy; regression uses MAE, RMSE, R².
- **Imbalanced classes:** do not rely on accuracy alone.
- **Outliers:** investigate context before removing them.
- **Limitations:** one split can be noisy; tune hyperparameters/cross-validate if time allows. Never repeatedly optimize using the test set.

### Common task adaptations
- **ANN question:** report the MLP row and explain network layers, activation, preprocessing, and evaluation.
- **Improve F1/recall:** examine class imbalance, use the appropriate metric, tune models and possibly decision thresholds.
- **Regression:** set `TASK_OVERRIDE = "regression"` and evaluate MAE/RMSE/R².
- **Classification:** set `TASK_OVERRIDE = "classification"` and report confusion matrix plus precision/recall/F1.
- **Forecasting:** use time-ordered splitting and time-based features; random shuffling can leak future information.
- **Clustering/no target:** use an unsupervised workflow such as K-Means and silhouette score; this notebook is mainly supervised.